In [41]:
from agent_framework import (
    tool,
    FunctionTool,
    FunctionInvocationContext,
)
import inspect
from typing import get_origin, Never, Annotated
from pydantic import Field, BaseModel

In [ ]:
from typing import TypedDict

class ToolParameterInfo(TypedDict):
    name: str
    type: list[type]
    required: bool
    default: object | None

class ToolInfo(TypedDict):
    name: str
    description: str
    approval_mode: str
    parameters: list[ToolParameterInfo]
    return_type: list[type]

In [ ]:
def is_function_tool(func) -> bool:
    # Note: issubclass works here, too
    return isinstance(type(func), FunctionTool)

In [ ]:
def _extract_types(tp) -> list[type]:
    if tp is None:
        return []
    
    origin = get_origin(tp)
    if origin in (typing.Union, types.UnionType):
        return [t for t in typing.get_args(tp)]
    return [tp]

In [ ]:
def get_functiontool_return_type(tool: FunctionTool) -> list[type]:
    if not is_function_tool(tool):
        raise ValueError("Provided function is not a FunctionTool.")
    
    hints = get_type_hints(tool.func)
    return _extract_types(hints.get("return", None))

In [ ]:
def get_functiontool_parameters(tool: FunctionTool) -> list[ToolParameterInfo]:
    if not is_function_tool(tool):
        raise ValueError("Provided function is not a FunctionTool.")
    


    parameters = tool.parameters()
    properties = parameters.get("properties", None)
    if properties is None:
        return []

    required = parameters.get("required", [])


    
    

    sig = inspect.signature(tool.func)
    hints = get_type_hints(tool.func)
    parameters_info = []
    
    for param_name, param in sig.parameters.items():
        param_type = hints.get(param_name, None)
        param_info = ToolParameterInfo(
            name=param_name,
            type=_extract_types(param_type),
            required=param.default is inspect.Parameter.empty,
            default=None if param.default is inspect.Parameter.empty else param.default
        )
        parameters_info.append(param_info)
    
    return parameters_info

In [ ]:
def get_functiontool_metadata(tool: FunctionTool) -> list[HandlerSpecInfo]:
    if not is_function_tool(tool):
        raise ValueError("Input must be an instance of FunctionTool")
    info: ToolInfo = {
        "name": tool.name,
        "description": tool.description,
        "approval_mode": tool.approval_mode,
        "output_type": get_functiontool_output_type(tool),
    }




    handler_info: list[HandlerSpecInfo] = []
    # Note: Use tool._original_func to access the original function
    for handler_spec in tool._handler_specs:
        name = handler_spec["name"]
        input_type = handler_spec["message_type"]
        output_types = handler_spec["output_types"]
        workflow_output_types = handler_spec["workflow_output_types"]

        handler_info.append({
            "handler_name": name,
            "input_type": input_type,
            "output_types": output_types,
            "workflow_output_types": workflow_output_types,
            "doc_string": get_handler_doc_string(tool, input_type), # Note: handlers require unique input types
            "input_parameter_names": get_handler_parameter_names(tool, input_type),
        })

    return handler_info

Scenarios to test:

- @tool()
- @tool(approval_mode, name, description)
- @tool(schema)
- class-based (tool decorator applied to a method)
- class-based (no tool decorator applied)

In [91]:
# Tool without approval mode
@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str | int, Field(description="The location to get the weather for.")],
    ctx: FunctionInvocationContext,
) -> str | int:
    """Get the weather for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C."

In [52]:
print(type(get_weather))

<class 'agent_framework._tools.FunctionTool'>


In [58]:
dir(get_weather)

['DEFAULT_EXCLUDE',
 'INJECTABLE',
 '_SHALLOW_COPY_FIELDS',
 '__annotations__',
 '__call__',
 '__class__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__get__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slotnames__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_cached_parameters',
 '_context_parameter_name',
 '_discover_injected_parameters',
 '_get_type_identifier',
 '_input_model_explicitly_provided',
 '_input_schema',
 '_input_schema_cached',
 '_instance',
 '_invocation_duration_histogram',
 '_is_context_parameter',
 '_make_dumpable',
 '_resolve_input_model',
 '_schema_supplied',
 'additional_properties',
 'approval_mode',
 'declaration_only',
 'description',
 'from_dict',
 '

In [45]:
# Tool without approval mode
@tool(name="weather_tool", description="Get the weather for a location", approval_mode="never_require")
def get_weather_approval(
    location: Annotated[str, Field(description="The location to get the weather for.")],
    ctx: FunctionInvocationContext,
) -> str:
    """Get the weather for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C."

In [46]:
class WeatherTool(BaseModel):
    location: Annotated[str, Field(description="The location to get the weather for.")]

# Tool without approval mode
@tool(schema=WeatherTool)
def get_weather_schema(
    location: str,
    ctx: FunctionInvocationContext,
) -> str:
    """Get the weather for a given location."""
    return f"The weather in {location} is cloudy with a high of 15°C."

In [ ]:
class MyToolClassWithHandlers:
    @tool(name="weather_tool", description="Get the weather for a location", approval_mode="never_require")
    def get_weather_method(
        self,
        location: Annotated[str, Field(description="The location to get the weather for.")],
        ctx: FunctionInvocationContext,
    ) -> str:
        """Get the weather for a given location."""
        return f"The weather in {location} is cloudy with a high of 15°C."


<class 'agent_framework._tools.FunctionTool'>


In [ ]:
class MyToolClassWithoutHandlers:
    def divide(
        self,
        a: Annotated[float, Field(description="The dividend")],
        b: Annotated[float, Field(description="The divisor")],
        ctx: FunctionInvocationContext
    ) -> float:
        """Divide two numbers."""
        if b == 0:
            raise ValueError("Cannot divide by zero")
        return a / b

    def add(
        self,
        a: Annotated[float, Field(description="The first addend")],
        b: Annotated[float, Field(description="The second addend")],
        ctx: FunctionInvocationContext
    ) -> float:
        """Add two numbers."""
        return a + b

In [49]:
inspect.signature(get_weather)

ValueError: no signature found for builtin <agent_framework._tools.FunctionTool object at 0x0000024FBB90C190>

In [ ]:
get_weather.approval_mode

'never_require'

In [ ]:
get_weather.description

'Get the weather for a given location.'

In [92]:
get_weather.parameters()

{'properties': {'location': {'anyOf': [{'type': 'string'},
    {'type': 'integer'}],
   'description': 'The location to get the weather for.',
   'title': 'Location'}},
 'required': ['location'],
 'title': 'get_weather_input',
 'type': 'object'}

In [ ]:
inspect.getdoc(get_weather.func)

'Get the weather for a given location.'

In [59]:
dir(inspect.signature(get_weather.func))

['__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__replace__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__slots__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '_bind',
 '_bound_arguments_cls',
 '_hash_basis',
 '_parameter_cls',
 '_parameters',
 '_return_annotation',
 'bind',
 'bind_partial',
 'empty',
 'format',
 'from_callable',
 'parameters',
 'replace',
 'return_annotation']

In [86]:
inspect.signature(get_weather.func).return_annotation

str | int

In [93]:
get_weather.to_json()

'{"type": "function_tool", "name": "get_weather", "description": "Get the weather for a given location.", "approval_mode": "never_require", "invocation_count": 0, "invocation_exception_count": 0, "input_model": {"properties": {"location": {"anyOf": [{"type": "string"}, {"type": "integer"}], "description": "The location to get the weather for.", "title": "Location"}}, "required": ["location"], "title": "get_weather_input", "type": "object"}}'

In [87]:
# This is how we can get the actual types, not just a string representation 
import typing
import types

hints = typing.get_type_hints(get_weather.func)
return_type = hints.get("return", None)

origin = typing.get_origin(return_type)
args = typing.get_args(return_type)

if origin in (typing.Union, types.UnionType):   # handles both forms
    print(args)
else:
    print("Not a union:", return_type)


{'location': <class 'str'>, 'ctx': <class 'agent_framework._middleware.FunctionInvocationContext'>, 'return': str | int}
(<class 'str'>, <class 'int'>)


In [94]:
# LEFT-OFF: if Gemini suggests to use the string based version, then can we do the same with executors?